In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import pandas as pd
data = pd.read_csv("../../Data/adult.csv")
print(data.shape)
data.count()[1]

In [ ]:
from aif360.datasets import StandardDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
import matplotlib.patches as patches
from aif360.algorithms.preprocessing import Reweighing
#from packages import *
#from ml_fairness import *
import matplotlib.pyplot as plt
import seaborn as sns



from IPython.display import Markdown, display

In [ ]:
data

In [ ]:
def cc(x):
    return sum(x=='?')
data.apply(cc)

In [ ]:
data.loc[data.workclass == '?'].apply(cc)

In [ ]:
data.groupby(by='workclass')['hours.per.week'].mean()

In [ ]:
df = data[data.occupation !='?']

In [ ]:
df.loc[df['native.country']!='United-States','native.country'] = 'non_usa'

In [ ]:
df

In [ ]:
for i in df.columns:
    if type(df[i][1])== str:
        print(df[i].value_counts())

In [ ]:
df.columns

In [ ]:
import seaborn as sns
fig, ((a,b),(c,d),(e,f)) = plt.subplots(3,2,figsize=(15,20))
plt.xticks(rotation=45)
sns.countplot(df['workclass'],hue=df['income'],ax=f)
sns.countplot(df['relationship'],hue=df['income'],ax=b)
sns.countplot(df['marital.status'],hue=df['income'],ax=c)
sns.countplot(df['race'],hue=df['income'],ax=d)
sns.countplot(df['sex'],hue=df['income'],ax=e)
sns.countplot(df['native.country'],hue=df['income'],ax=a)

In [ ]:
fig, (a,b)= plt.subplots(1,2,figsize=(20,6))
sns.boxplot(y='hours.per.week',x='income',data=df,ax=a)
sns.boxplot(y='age',x='income',data=df,ax=b)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df_backup =df

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for i in df.columns:
    df[i]=le.fit_transform(df[i])
    print(df[i])
    
    


In [ ]:
import random
import sklearn
random.seed(100)
train,test = train_test_split(df,test_size=0.2)

In [ ]:
df

# Baseline model 

We always set a baseline model which determines the threshold accuracy. The accuracy above this is only selected. Unlike regression model for which the baseline accuracy is mean of all outcomes. The baseline accuracy for classification model is mode of target variable.

In [ ]:
l=pd.DataFrame(test['income'])
l['baseline'] =0
k = pd.DataFrame(sklearn.metrics.confusion_matrix(test['income'],l['baseline']))
print(k)
(k[0][0]+k[1][1])/(k[0][0]+k[1][1]+k[0][1]+k[1][0])

Hence our  baseline accuracy comes out to be 75%

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression()
x=train.drop('income',axis=1)
y=train['income']
clf.fit(x,y)

In [ ]:
clf.score(x,y)

In [ ]:
pred = clf.predict(test.drop('income',axis=1))

# Note this our Confusion Matrix

In [ ]:
import sklearn
k = pd.DataFrame(sklearn.metrics.confusion_matrix(test['income'],pred))
print(k)

# precision of the model

In [ ]:
(k[0][0]+k[1][1])/(k[0][0]+k[1][1]+k[0][1]+k[1][0])

In [ ]:
y_score = clf.fit(x,y).decision_function(test.drop('income',axis=1))

fpr,tpr,the=sklearn.metrics.roc_curve(test['income'],y_score)
sklearn.metrics.roc_auc_score(test['income'],pred)
plt.plot(fpr,tpr,)

# Area under ROC curve.
 which comes out pretty well !

In [ ]:
sklearn.metrics.roc_auc_score(test['income'],y_score)

In [ ]:
col=['age','fnlwgt','capital.gain','capital.loss','hours.per.week','education','education.num','marital.status','relationship','sex']

# PCA analysis
#### Principal Component Analysis (or PCA) uses linear algebra to transform the dataset into a compressed form.

#### Generally this is called a data reduction technique. A property of PCA is that you can choose the number of dimensions or principal component in the transformed result.

#### In this case we will use it to analyse the feature importanace

In [ ]:
from sklearn.preprocessing import StandardScaler
X_std = StandardScaler().fit_transform(x)


from sklearn.decomposition import PCA as sklearnPCA
sklearn_pca = sklearnPCA(n_components=14)
Y_sklearn = sklearn_pca.fit_transform(X_std)

cum_sum = sklearn_pca.explained_variance_ratio_.cumsum()

sklearn_pca.explained_variance_ratio_[:10].sum()

cum_sum = cum_sum*100

fig, ax = plt.subplots(figsize=(8,8))
plt.bar(range(14), cum_sum, label='Cumulative _Sum_of_Explained _Varaince', color = 'b',alpha=0.5)

## Results:
As you can see every feature adds significant variance hence droping the variables in this case is not a good idea!
Therefore we should use every feature present to build our model as done previously.

## Decision Tree Model.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

## Tuning the parameters of decision tree.
max_features =10  (from above cum_sum chart) <br>
min_samples_leaf=100 controling the branches by setting the limits (Pruning)

In [ ]:
clf = DecisionTreeClassifier(max_features=14,min_samples_leaf=100,random_state=10)
clf.fit(x,y)

In [ ]:
pred2 = clf.predict(test.drop('income',axis=1))

In [ ]:
k = pd.DataFrame(sklearn.metrics.confusion_matrix(test['income'],pred2))
print(k)

In [ ]:
(k[0][0]+k[1][1])/(k[0][0]+k[1][1]+k[0][1]+k[1][0])

# Nice result!

# Now comes Xgboost

In [ ]:
from xgboost import XGBClassifier

clf= XGBClassifier()

clf.fit(x,y)

pred2 = clf.predict(test.drop('income',axis=1))

k = pd.DataFrame(sklearn.metrics.confusion_matrix(test['income'],pred2))


(k[0][0]+k[1][1])/(k[0][0]+k[1][1]+k[0][1]+k[1][0])

86.7% accuracy! quite good.

In [ ]:
from catboost import CatBoostClassifier

clf= CatBoostClassifier(learning_rate=0.04)

clf.fit(x,y)

pred2 = clf.predict(test.drop('income',axis=1))

k = pd.DataFrame(sklearn.metrics.confusion_matrix(test['income'],pred2))


(k[0][0]+k[1][1])/(k[0][0]+k[1][1]+k[0][1]+k[1][0])

In [ ]:
clf.score(x,y)

## Fairness

In [ ]:
# This DataFrame is created to stock differents models and fair metrics that we produce in this notebook
algo_metrics = pd.DataFrame(columns=['model', 'fair_metrics', 'prediction', 'probs'])

def add_to_df_algo_metrics(algo_metrics, model, fair_metrics, preds, probs, name):
    return algo_metrics.append(pd.DataFrame(data=[[model, fair_metrics, preds, probs]], columns=['model', 'fair_metrics', 'prediction', 'probs'], index=[name]))

In [ ]:
def fair_metrics(dataset, pred, pred_is_dataset=False):
    if pred_is_dataset:
        dataset_pred = pred
    else:
        dataset_pred = dataset.copy()
        dataset_pred.labels = pred
    
    cols = ['statistical_parity_difference', 'equal_opportunity_difference', 'average_abs_odds_difference',  'disparate_impact', 'theil_index']
    obj_fairness = [[0,0,0,1,0]]
    
    fair_metrics = pd.DataFrame(data=obj_fairness, index=['objective'], columns=cols)
    
    for attr in dataset_pred.protected_attribute_names:
        idx = dataset_pred.protected_attribute_names.index(attr)
        privileged_groups =  [{attr:dataset_pred.privileged_protected_attributes[idx][0]}] 
        unprivileged_groups = [{attr:dataset_pred.unprivileged_protected_attributes[idx][0]}] 
        
        classified_metric = ClassificationMetric(dataset, 
                                                     dataset_pred,
                                                     unprivileged_groups=unprivileged_groups,
                                                     privileged_groups=privileged_groups)

        metric_pred = BinaryLabelDatasetMetric(dataset_pred,
                                                     unprivileged_groups=unprivileged_groups,
                                                     privileged_groups=privileged_groups)

        acc = classified_metric.accuracy()

        row = pd.DataFrame([[metric_pred.mean_difference(),
                                classified_metric.equal_opportunity_difference(),
                                classified_metric.average_abs_odds_difference(),
                                metric_pred.disparate_impact(),
                                classified_metric.theil_index()]],
                           columns  = cols,
                           index = [attr]
                          )
        fair_metrics = fair_metrics.append(row)    
    
    fair_metrics = fair_metrics.replace([-np.inf, np.inf], 2)
        
    return fair_metrics

def plot_fair_metrics(fair_metrics):
    fig, ax = plt.subplots(figsize=(20,4), ncols=5, nrows=1)

    plt.subplots_adjust(
        left    =  0.125, 
        bottom  =  0.1, 
        right   =  0.9, 
        top     =  0.9, 
        wspace  =  .5, 
        hspace  =  1.1
    )

    y_title_margin = 1.2

    plt.suptitle("Fairness metrics", y = 1.09, fontsize=20)
    sns.set(style="dark")

    cols = fair_metrics.columns.values
    obj = fair_metrics.loc['objective']
    size_rect = [0.2,0.2,0.2,0.4,0.25]
    rect = [-0.1,-0.1,-0.1,0.8,0]
    bottom = [-1,-1,-1,0,0]
    top = [1,1,1,2,1]
    bound = [[-0.1,0.1],[-0.1,0.1],[-0.1,0.1],[0.8,1.2],[0,0.25]]

    display(Markdown("### Check bias metrics :"))
    display(Markdown("A model can be considered bias if just one of these five metrics show that this model is biased."))
    for attr in fair_metrics.index[1:len(fair_metrics)].values:
        display(Markdown("#### For the %s attribute :"%attr))
        check = [bound[i][0] < fair_metrics.loc[attr][i] < bound[i][1] for i in range(0,5)]
        display(Markdown("With default thresholds, bias against unprivileged group detected in **%d** out of 5 metrics"%(5 - sum(check))))

    for i in range(0,5):
        plt.subplot(1, 5, i+1)
        ax = sns.barplot(x=fair_metrics.index[1:len(fair_metrics)], y=fair_metrics.iloc[1:len(fair_metrics)][cols[i]])
        
        for j in range(0,len(fair_metrics)-1):
            a, val = ax.patches[j], fair_metrics.iloc[j+1][cols[i]]
            marg = -0.2 if val < 0 else 0.1
            ax.text(a.get_x()+a.get_width()/5, a.get_y()+a.get_height()+marg, round(val, 3), fontsize=15,color='black')

        plt.ylim(bottom[i], top[i])
        plt.setp(ax.patches, linewidth=0)
        ax.add_patch(patches.Rectangle((-5,rect[i]), 10, size_rect[i], alpha=0.3, facecolor="green", linewidth=1, linestyle='solid'))
        plt.axhline(obj[i], color='black', alpha=0.3)
        plt.title(cols[i])
        ax.set_ylabel('')    
        ax.set_xlabel('')

In [ ]:
def get_fair_metrics_and_plot(data, model, plot=False, model_aif=False):
    pred = model.predict(data).labels if model_aif else model.predict(data.features)
    # fair_metrics function available in the metrics.py file
    fair = fair_metrics(data, pred)

    if plot:
        # plot_fair_metrics function available in the visualisations.py file
        # The visualisation of this function is inspired by the dashboard on the demo of IBM aif360 
        plot_fair_metrics(fair)
        display(fair)
    
    return fair

In [ ]:
df.head()

In [ ]:
#print(X)


#combine_final = [train_df, test_df]
#result = pd.concat(combine_final)
#print(result.ifany())
#print(result)
privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]
dataset_orig = StandardDataset(df,
                                  label_name='income',
                                  protected_attribute_names=['sex'],
                                  favorable_classes=[1],
                                  privileged_classes=[[1]])

#metric_orig_train = BinaryLabelDatasetMetric(dataset_orig, 
#                                             unprivileged_groups=unprivileged_groups,
#                                             privileged_groups=privileged_groups)
#display(Markdown("#### Original training dataset"))
#print("Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_orig_train.mean_difference())


In [ ]:
metric_orig_train = BinaryLabelDatasetMetric(dataset_orig, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
print("Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_orig_train.mean_difference())

In [ ]:
data_orig_train = pd.read_pickle('../../Results/XGBoost/7-deep-analysis-and-90-accuracy_Train.pkl')
data_orig_test = pd.read_pickle('../../Results/XGBoost/7-deep-analysis-and-90-accuracy_Test.pkl')

In [ ]:
import ipynbname
nb_fname = ipynbname.name()
nb_path = ipynbname.path()

from xgboost import XGBClassifier
import pickle

#data_orig_train, data_orig_test = dataset_orig.split([0.7], shuffle=True)
X_train = data_orig_train.features
y_train = data_orig_train.labels.ravel()

X_test = data_orig_test.features
y_test = data_orig_test.labels.ravel()
num_estimators = 100

model = XGBClassifier(n_estimators= 1)

mdl = model.fit(X_train, y_train)
#with open('../../Results/XGBoost/' + nb_fname + '.pkl', 'wb') as f:
#        pickle.dump(mdl, f)

#with open('../../Results/XGBoost/' + nb_fname + '_Train' + '.pkl', 'wb') as f:
#    pickle.dump(data_orig_train, f) 
    
#with open('../../Results/XGBoost/' + nb_fname + '_Test' + '.pkl', 'wb') as f:
#    pickle.dump(data_orig_test, f) 

In [ ]:
from csv import writer
from sklearn.metrics import accuracy_score, f1_score

final_metrics = []
accuracy = []
f1= []

for i in range(1,num_estimators+1):
    
    model = XGBClassifier(n_estimators= i, colsample_bytree=0.3, gamma= 0.1, learning_rate= 0.15, max_depth= 10, min_child_weight= 5, booster = 'dart', rate_drop = 0.5)
    mdl = model.fit(X_train, y_train)
    yy = mdl.predict(X_test)
    accuracy.append(accuracy_score(y_test, yy))
    f1.append(f1_score(y_test, yy))
    fair = get_fair_metrics_and_plot(data_orig_test, mdl)                           
    fair_list = fair.iloc[1].tolist()
    #fair_list.insert(0, i)
    final_metrics.append(fair_list)


In [ ]:
import numpy as np
final_result = pd.DataFrame(final_metrics)
#print(final_result)
final_result[3] = np.log(final_result[3])
final_result = final_result.transpose()
acc_f1 = pd.DataFrame(accuracy)
acc_f1['f1'] = f1
acc_f1 = pd.DataFrame(acc_f1).transpose()
acc = acc_f1.rename(index={0: 'accuracy', 1: 'f1'})
final_result = final_result.rename(index={0: 'statistical_parity_difference', 1: 'equal_opportunity_difference', 2: 'average_abs_odds_difference', 3: 'disparate_impact', 4: 'theil_index'})
final_result = pd.concat([acc,final_result])
final_result.columns = ['T' + str(col) for col in final_result.columns]
final_result.insert(0, "classifier", final_result['T' + str(num_estimators - 1)])   ##Add final metrics add the beginning of the df
#final_result.to_csv('../../Results/XGBoost/' + nb_fname + '.csv')
final_result

In [ ]:
list(final_result['classifier'])
